# Controlling an interferometer using EPICS and a stage using ophyd
awojdyla@lbl.gov, Feb 2026

In [1]:
# !pip install pyepics
# !pip install ophyd

In [2]:
#using base 3.13.5 on GW
import epics
import ophyd

## Reading the PV for the interferometer
We assume that an IOC is running for the Smaract interferomter, and that we can poll it through the network

In [3]:
epics.caget('CATERETE:PICOSCALE:POS_0')

-171677079

## Moving the stage

### using the python package

In [4]:
#!pip install newportxps

### create an ophyd wrapper

In [1]:
from bluesky import RunEngine

import bluesky.plans as bp
import bluesky.plan_stubs as bps

# Create a Run Engine
RE = RunEngine()

In [2]:
from ophyd import Device, Component as Cpt, Signal
from ophyd import EpicsSignal
from ophyd.status import SubscriptionStatus
import threading
import time

class NewportXPSMotor(Device):
    # Ophyd signals
    user_readback = Cpt(Signal, value=0, kind='hinted')
    user_setpoint = Cpt(Signal, value=0, kind='normal')
    
    def __init__(self, ip_address, stage_name, username='Administrator', 
                 password='Administrator', *args, **kwargs):
        super().__init__(*args, **kwargs)
        
        from newportxps import NewportXPS
        
        self.stage_name = stage_name
        self.group_name = stage_name.split('.')[0]  # Extract 'Group1' from 'Group1.Pos'
        self.xps = NewportXPS(ip_address, username=username, password=password)
        
        # Update initial position
        self._update_position()
        
    def _update_position(self):
        """Read current position from hardware"""
        pos = self.xps.read_stage_position(self.stage_name)
        self.user_readback.put(pos)
        return pos
    
    def _is_moving(self):
        """Check if the motor is currently moving"""
        status = self.xps.get_group_status()
        group_status = status.get(self.group_name, '')
        return 'Moving' in group_status
    
    def set(self, position, timeout=30):
        """Move to a position and return a status object"""
        
        def check_done(*, old_value, value, **kwargs):
            """Callback to check if motion is complete"""
            return not self._is_moving()
        
        # Start the move
        self.user_setpoint.put(position)
        self.xps.move_stage(self.stage_name, position)
        
        # Create a status object that monitors completion
        status = SubscriptionStatus(self.user_readback, check_done, timeout=timeout)
        
        # Start a background thread to update position during motion
        def monitor_motion():
            while not status.done:
                self._update_position()
                time.sleep(0.05)  # 50ms polling
            # Final position update
            self._update_position()
        
        thread = threading.Thread(target=monitor_motion, daemon=True)
        thread.start()
        
        return status
    
    def read(self):
        """Read current position"""
        pos = self._update_position()
        return {f'{self.name}_user_readback': {'value': pos, 'timestamp': time.time()}}
    
    def describe(self):
        """Describe the readback signal"""
        return {f'{self.name}_user_readback': {
            'source': f'XPS:{self.stage_name}',
            'dtype': 'number',
            'shape': []
        }}

In [6]:
#stock ophyd opbject for EPICS signal
interf = EpicsSignal('CATERETE:PICOSCALE:POS_0', name='interferometer_position')

#custom ophyd object
motor = NewportXPSMotor(
    ip_address='192.168.10.20',
    stage_name='Group1.Pos',
    username='Administrator',
    password='Administrator',
    name='xps_motor'
)

pos_inter_m = interf.get()*1e-12
pos_readback_m = motor.get()[0]*1e-3
print(f"Interferometer position: {pos_inter_m*1e3} mm")
print(f"Motor readback position: {pos_readback_m*1e3} mm")


Interferometer position: -1.795185102 mm
Motor readback position: 25.000002000000002 mm


In [10]:
from tiled.client import from_uri
import os
from bluesky.callbacks.tiled_writer import TiledWriter

#import os
#api_key = os.getenv("TILED_SINGLE_USER_API_KEY")

api_key_filepath = "C:\\Users\\admin\\Documents\\TILED_SINGLE_USER_API_KEY.txt"
with open(api_key_filepath, 'r') as f:
    api_key = f.read().strip()
os.environ["TILED_SINGLE_USER_API_KEY"] = api_key

if not api_key:
   raise ValueError("TILED_SINGLE_USER_API_KEY environment variable is not set.")
#from bluesky.callbacks.tiled_writer import TiledWriter

tiled_client = from_uri("http://131.243.80.226:8000/", api_key=api_key)
tw = TiledWriter(tiled_client)

from bluesky import RunEngine
RE = RunEngine()
RE.subscribe(tw)


#RE(bp.scan([interf, motor], motor, -10, 10, 11))
RE(bp.count([interf], 11))

Run aborted
Traceback (most recent call last):
  File "c:\Users\admin\anaconda3\Lib\site-packages\tiled\client\utils.py", line 74, in handle_error
    raise_for_status(response)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "c:\Users\admin\anaconda3\Lib\site-packages\tiled\client\utils.py", line 67, in raise_for_status
    raise httpx.HTTPStatusError(message, request=request, response=response)
httpx.HTTPStatusError: Client error '422 Unprocessable Entity' for url 'http://131.243.80.226:8000/api/v1/metadata//22fedf78-f93e-4048-859e-24e367c7c7bf/primary'
For more information, server admin can search server logs for correlation ID 122e5ee4c81dea76.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\admin\anaconda3\Lib\site-packages\bluesky\run_engine.py", line 1610, in _run
    msg = self._plan_stack[-1].send(resp)
  File "c:\Users\admin\anaconda3\Lib\site-packages\bluesky\plans.py", line 129, in count
    return (yield from

ClientError: 422: [{'type': 'extra_forbidden', 'loc': ['body', 'data_sources', 0, 'properties'], 'msg': 'Extra inputs are not permitted', 'input': {}}] http://131.243.80.226:8000/api/v1/metadata//22fedf78-f93e-4048-859e-24e367c7c7bf/primary

In [ ]:
from tiled.client import simple
c = simple('../data/test')  # or just simple() for temporary/dev storage
#ifndef 
# c.create_container('raw', specs=['CatalogOfBlueskyRuns'])

from bluesky_tiled_plugins import TiledWriter
tw = TiledWriter(c['raw'])

from bluesky import RunEngine
RE = RunEngine()
RE.subscribe(tw)


RE(bp.scan([interf, motor], motor, -10, 10, 11))

Tiled version 0.2.6


http://127.0.0.1:53946/api/v1?api_key=88abd975d811bd7a


Scheduled retry in 0.87 seconds due to HTTPStatusError("Server error '500 Internal Server Error' for url 'http://127.0.0.1:53946/api/v1/metadata//db6ac1c7-ea50-4cab-8d06-75cf05fbda62/primary'\nFor more information, server admin can search server logs for correlation ID 2ffa28ec6ee01b21.") (attempt 1)
Run aborted
Traceback (most recent call last):
  File "c:\Users\admin\anaconda3\Lib\site-packages\tiled\client\utils.py", line 74, in handle_error
    raise_for_status(response)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "c:\Users\admin\anaconda3\Lib\site-packages\tiled\client\utils.py", line 67, in raise_for_status
    raise httpx.HTTPStatusError(message, request=request, response=response)
httpx.HTTPStatusError: Client error '409 Conflict' for url 'http://127.0.0.1:53946/api/v1/metadata//db6ac1c7-ea50-4cab-8d06-75cf05fbda62/primary'
For more information, server admin can search server logs for correlation ID b2abd272c3d31d8a.

The above exception was the direct cause of the following exceptio

ClientError: 409: /db6ac1c7-ea50-4cab-8d06-75cf05fbda62/primary/internal http://127.0.0.1:53946/api/v1/metadata//db6ac1c7-ea50-4cab-8d06-75cf05fbda62/primary

That didn't work:

In [15]:
# from bluesky.callbacks.tiled_writer import TiledWriter
# from envs.p312.Lib import http
# from tiled.client import from_uri

# # Connect to your local server
# #tiled_client = from_uri("http://127.0.0.1:8000?api_key=b762dd7884cae5dd192841af420db11edc3dc16001b7b6c948945e82c0075596") 
# tiled_client = from_uri('http://192.168.10.240:8000?api_key=e73d7fa0c47f4b9f1a230526e1d3d669d6dcdd06ce96ab0391a00f431347de69')


# # from bluesky.callbacks.tiled_writer import TiledWriter
# # tiled_writer = TiledWriter(tiled_server)

# # Initialize and subscribe
# tw = TiledWriter(tiled_client)
# RE.subscribe(tw)

# RE(bp.scan([interf, motor], motor, -10, 10, 11))

## using ALS bl 5.3.1 tiled server